# Cleaning

In [ ]:
# Import, clean

import pandas as pd

games = pd.read_csv("data/games.csv")
matches = pd.read_csv("data/matches.csv")
tournaments = pd.read_csv("data/tour_meta.csv")

# Clean
# drop missing dates (games)
games['date'] = pd.to_datetime(games.date.str.split('(').str[0].str.strip())
games = games[pd.notna(games['date'])]
# sort by date
games.sort_values('date', inplace=True)

# clean dates (matches)
matches['date'] = pd.to_datetime(matches.date)
matches = matches[pd.notna(matches['date'])]
# sort by date
matches.sort_values('date', inplace=True)
# drop nan scores
matches = matches[pd.notna(matches['team1_score']) & pd.notna(matches['team2_score'])]

In [ ]:
# Tournament attributes

import re
import pandas as pd

def map_tournament_attributes(matches: pd.DataFrame) -> pd.DataFrame:

    def get_region(name: str) -> str:
        if name.startswith("LEC"):
            return "LEC"
        if name.startswith(("LCK", "LCK Cup")):
            return "LCK"
        if name.startswith(("LPL", "LPL ")):
            return "LPL"
        if name.startswith(("LCS", "LTA", "LCP")):
            return "LCS"
        if name.startswith(("MSI", "Mid-Season Invitational")):
            return "INTL"
        if "World" in name or "Esports World Cup" in name:
            return "INTL"
        return "OTHER"

    def get_season(name: str) -> int | None:
        m = re.search(r"(20\d{2})", name)  # pl 2023/2024/2025
        return int(m.group(1)) if m else None

    def get_split(name: str) -> str:
        # Normalize split: Spring/Winter → Split1, Summer/Split2 → Split2, MSI/Worlds/EWC → International
        lowered = name.lower()
        if any(x in lowered for x in ["msi", "world", "esports world cup"]):
            return "International"
        if any(x in lowered for x in ["winter", "spring", "split 1"]):
            return "Split1"
        if any(x in lowered for x in ["summer", "split 2"]):
            return "Split2"
        return "Other"

    def get_stage(name: str) -> str:
        lowered = name.lower()
        if "groups" in lowered:
            return "Groups"
        if "playoffs" in lowered:
            return "Playoffs"
        if "regional finals" in lowered:
            return "Regional Finals"
        if "championship" in lowered:
            return "Championship"
        if "play-in" in lowered or "qualifying" in lowered:
            return "Qualifier"
        if "main event" in lowered:
            return "Main Event"
        # alap esetben Season
        if any(x in lowered for x in ["season", "split", "rounds"]):
            return "Season"
        return "Unknown"

    def is_international(name: str) -> int:
        lowered = name.lower()
        return 1 if any(x in lowered for x in ["msi", "world", "esports world cup"]) else 0

    def is_regional_final(name: str) -> int:
        return 1 if "regional finals" in name.lower() else 0

    matches["region"] = matches["tournament_name"].apply(get_region)
    matches["season"] = matches["tournament_name"].apply(get_season)
    matches["split"] = matches["tournament_name"].apply(get_split)
    matches["stage"] = matches["tournament_name"].apply(get_stage)
    matches["is_international"] = matches["tournament_name"].apply(is_international)
    matches["is_regional_final"] = matches["tournament_name"].apply(is_regional_final)

    return matches


tournaments = map_tournament_attributes(tournaments)
display(tournaments.sample(5))
print(tournaments[['tournament_name', 'region', 'season', 'split', 'stage', 'is_international', 'is_regional_final']].info())

In [ ]:
# ELO calc

import pandas as pd

matches = pd.merge(matches, tournaments[['tournament_name', 'region', 'is_international']], 
                   how='left', on='tournament_name')

# kezdő ELO minden csapatnak
initial_elo = 1500
# példa: ligánként eltérő kezdő ELO
league_base_elo = {
    "LPL": 1550,
    "LCK": 1550,
    "LEC": 1525,
    "LCS": 1500,
    "LTA": 1450,
    "LCP": 1450,
    "MSI": 1580,  # ha nemzetközi bónusz
    "Worlds": 1600
}
K = 20  # standard K érték, meccsenként

# létrehozunk egy dictionary-t a csapatok ELO-jához
team_elos = {}

# új oszlopok a matches-hez
matches['team1_elo_before'] = 0
matches['team2_elo_before'] = 0
matches['team1_elo_after'] = 0
matches['team2_elo_after'] = 0

# Liga matchup korrekció: international meccsekből számoljuk
international_matches = matches[matches.get('is_international', 0) == 1]

# Liga matchup dict létrehozása
league_correction = {}  # pl. ('LPL','LEC') -> +20
for idx, row in international_matches.iterrows():
    l1, l2 = row['region'], row['region']  # ha külön kell: team1_region, team2_region
    l1, l2 = row['team1_region'] if 'team1_region' in row else l1, row['team2_region'] if 'team2_region' in row else l2
    key = (l1, l2)
    league_correction.setdefault(key, []).append(1 if row['team1_score'] > row['team2_score'] else 0)

# Átlagos korrekció számítása
for k in league_correction:
    # winrate: 0-1 → átalakítjuk Elo pontban, pl. *100
    league_correction[k] = (sum(league_correction[k]) / len(league_correction[k]) - 0.5) * 200  

# rendezzük időrendbe
matches = matches.sort_values('date').reset_index(drop=True)

for idx, row in matches.iterrows():
    t1 = row['team1']
    t2 = row['team2']
    league1 = row['region']
    league2 = row['region']
    
    if t1 not in team_elos:
        base = league_base_elo.get(league1, initial_elo)
        team_elos[t1] = base
    if t2 not in team_elos:
        base = league_base_elo.get(league2, initial_elo)
        team_elos[t2] = base

    elo1 = team_elos[t1]
    elo2 = team_elos[t2]

    matches.at[idx, 'team1_elo_before'] = elo1
    matches.at[idx, 'team2_elo_before'] = elo2

    # Liga matchup korrekció hozzáadása
    correction = league_correction.get((league1, league2), 0)
    
    expected1 = 1 / (1 + 10 ** (((elo2 - elo1) - correction) / 400))
    expected2 = 1 / (1 + 10 ** (((elo1 - elo2) + correction) / 400))

    if row['team1_score'] > row['team2_score']:
        s1, s2 = 1, 0
    else:
        s1, s2 = 0, 1

    elo1_new = elo1 + K * (s1 - expected1)
    elo2_new = elo2 + K * (s2 - expected2)

    team_elos[t1] = elo1_new
    team_elos[t2] = elo2_new

    matches.at[idx, 'team1_elo_after'] = elo1_new
    matches.at[idx, 'team2_elo_after'] = elo2_new

team_elos

In [ ]:
# ELO visualization

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------
# 1️⃣ ELO rangsor jelenleg
# -------------------------------
# Kiszámítjuk a végső ELO-t minden csapatra (utolsó meccs után)
elo_latest_team1 = matches.groupby('team1')['team1_elo_after'].last()
elo_latest_team2 = matches.groupby('team2')['team2_elo_after'].last()

elo_latest = pd.concat([elo_latest_team1, elo_latest_team2], axis=1)
elo_latest['final_elo'] = elo_latest.max(axis=1)
elo_rank = elo_latest['final_elo'].sort_values(ascending=False)
print("=== Jelenlegi ELO rangsor ===")
print(elo_rank.head(10))

# -------------------------------
# 2️⃣ Upset / meglepetés meccsek
# -------------------------------
# Definiáljuk, hogy upset, ha a győztes ELO-ja legalább 100 ponttal kevesebb volt a vesztesnél
def detect_upset(row):
    if row['team1_score'] > row['team2_score'] and row['team1_elo_after'] + 100 < row['team2_elo_after']:
        return True
    elif row['team2_score'] > row['team1_score'] and row['team2_elo_after'] + 100 < row['team1_elo_after']:
        return True
    else:
        return False

matches['upset'] = matches.apply(detect_upset, axis=1)
num_upsets = matches['upset'].sum()
print(f"Number of upsets: {num_upsets}")

# -------------------------------
# 3️⃣ Ligánkénti ELO trend
# -------------------------------
plt.figure(figsize=(14,6))
sns.lineplot(data=matches, x='date', y='team1_elo_after', hue='team1', alpha=0.5)
plt.title("ELO trend csapatonként ligánként (team1 ELO)")
plt.xlabel("Date")
plt.ylabel("ELO")
plt.legend([],[], frameon=False)  # túl sok csapat, ezért elrejtjük a legend-et
plt.show()

# -------------------------------
# 4️⃣ ELO boxplot csapatonként
# -------------------------------
plt.figure(figsize=(16,6))
top_teams = matches['team1'].value_counts().nlargest(10).index  # Top 10 gyakori csapat
sns.boxplot(data=matches[matches['team1'].isin(top_teams)], x='team1', y='team1_elo_after')
plt.title("ELO boxplot csapatonként (top 10 gyakori csapat)")
plt.xlabel("Team")
plt.ylabel("ELO")
plt.xticks(rotation=45)
plt.show()

# Odds

## OddsPortal

In [ ]:
# Tournaments

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

def scrape_oddsportal_lol_links():
    url = "https://www.oddsportal.com/results/#esports"
    driver = webdriver.Chrome()
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    # várjuk, amíg betöltődik legalább 1 tournament link
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")))

    time.sleep(2)  # kis extra wait, hogy minden JS lefusson

    links = driver.find_elements(By.CSS_SELECTOR, "li[data-testid='results-tournament-box'] a")

    data = []
    for link in links:
        try:
            text = link.text.strip()
            href = link.get_attribute("href")
            if text.startswith("League of Legends "):
                name = text.replace("League of Legends ", "").strip()
                data.append({
                    "Name": name,
                    "url": href
                })
        except Exception as e:
            print(f"⚠️ Hiba egy link feldolgozásánál: {e}")
            continue

    driver.quit()
    df = pd.DataFrame(data)
    print(f"✅ {len(df)} League of Legends esemény található az OddsPortalon.")
    return df


# --- Példa futtatás ---
df_oddsportal = scrape_oddsportal_lol_links()
display(df_oddsportal)


In [ ]:
# Odds of tournament

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re, time
from datetime import datetime

def scrape_oddsportal_fixed(event_url, headless=False):
    opts = webdriver.ChromeOptions()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("user-agent=Mozilla/5.0")
    
    driver = webdriver.Chrome(options=opts)
    driver.get(event_url)
    wait = WebDriverWait(driver, 20)

    # --- Cookie gomb elfogadása, ha van ---
    try:
        cookie_btn = wait.until(EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler")))
        cookie_btn.click()
        time.sleep(1)
        print("🍪 Cookie elfogadva")
    except:
        pass

    # --- 2023 és 2024 szezon linkek lekérdezése ---
    season_links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/results/']")
    season_urls = []
    season_urls.append(event_url)
    for a in season_links:
        href = a.get_attribute("href")
        if re.search(r'2023/results/', href) or re.search(r'2024/results/', href):
            season_urls.append(href)

    # Ha nincs külön link, használjuk az eredeti event_url-t
    if not season_urls:
        season_urls = [event_url]

    all_data = []

    # --- Iterálunk a szezon linkeken ---
    for season_url in season_urls:
        driver.get(season_url)
        time.sleep(2)
        print(f"📅 Szezon feldolgozása: {season_url}")

        page = 1
        while True:
            print(f"🔍 Oldal {page} feldolgozása...")
            
            # Inkrementális scroll: lépésekben, lassabban
            scroll_pause = 1.5
            scroll_step = 500  # pixelenként
            last_height = driver.execute_script("return document.body.scrollHeight")
            current_pos = 0

            while current_pos < last_height:
                driver.execute_script(f"window.scrollTo(0, {current_pos});")
                time.sleep(scroll_pause)
                current_pos += scroll_step
                new_height = driver.execute_script("return document.body.scrollHeight")
                if new_height > last_height:
                    last_height = new_height


            time.sleep(2)
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.eventRow")))

            event_rows = driver.find_elements(By.CSS_SELECTOR, "div.eventRow")
            current_date = None

            for event in event_rows:
                # dátum keresése
                date_found = False
                
                # Dátum keresése az event teljes szövegében
                date_text = event.text.strip()
                if any(month in date_text for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                    lines = date_text.split('\n')
                    for line in lines:
                        if any(month in line for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']):
                            # Dátum formázása - eltávolítjuk a " -" utáni részt
                            clean_date = line.split(' -')[0].strip()
                            current_date = clean_date
                            
                            # Dátum átalakítása YYYY-MM-dd formátumba
                            try:
                                date_obj = datetime.strptime(current_date, '%d %b %Y')
                                current_date = date_obj.strftime('%Y-%m-%d')
                            except ValueError:
                                # Ha nem sikerül átalakítani, marad az eredeti
                                pass
                            
                            date_found = True
                            break

                # Ha nincs dátum, de van game-row, akkor meccs sor
                try:
                    game_row = event.find_element(By.CSS_SELECTOR, "div[data-testid='game-row']")
                except:
                    continue

                # csapatnevek
                participants = game_row.find_elements(By.CSS_SELECTOR, "a[title]")
                if len(participants) < 2:
                    continue

                home_team = participants[0].get_attribute("title").strip()
                away_team = participants[1].get_attribute("title").strip()

                # oddsok
                odds_blocks = event.find_elements(By.CSS_SELECTOR, "div[data-testid^='odd-container'] p")
                odds = []
                for p in odds_blocks:
                    try:
                        odds_text = p.text.strip().replace(",", ".")
                        if re.match(r"^\d+(\.\d+)?$", odds_text):
                            odds.append(float(odds_text))
                    except:
                        continue
                
                home_odds = odds[0] if len(odds) > 0 else None
                away_odds = odds[1] if len(odds) > 1 else None

                all_data.append({
                    "Date": current_date,
                    "home_team": home_team,
                    "away_team": away_team,
                    "home_odds": home_odds,
                    "away_odds": away_odds
                })

            # Következő oldal ellenőrzése
            try:
                next_button = driver.find_element(By.CSS_SELECTOR, f"a.pagination-link[data-number='{page + 1}']")
                if next_button.is_enabled():
                    print(f"➡️ Következő oldal: {page + 1}")
                    driver.execute_script("arguments[0].click();", next_button)
                    page += 1
                    time.sleep(3)  # Várakozás az oldal betöltésére
                    continue
                else:
                    break
            except:
                # Ha nincs következő oldal, kilépünk
                break

    driver.quit()
    
    df = pd.DataFrame(all_data).drop_duplicates(subset=["home_team","away_team","home_odds","away_odds"])
    print(f"✅ Összesen {len(df)} meccs feldolgozva {page} oldalról.")
    return df

# Teszt
#url = "https://www.oddsportal.com/esports/league-of-legends/league-of-legends-world-championship/results/"
#df_odds = scrape_oddsportal_fixed(url, headless=False)
#display(df_odds)

In [ ]:
# Relevant odds
  
oddsportal_relevant = [
    "LEC",
    "LCS Lock-In",
    "LTA North",
    "LTA South",
    "LTA Cross Conference",
    "LCP",
    "LCK",
    "LPL",
    "World Championship",
    "Esports World Cup"
]

df_odds_super = pd.DataFrame()
for op_tour in oddsportal_relevant:
    url_op = df_oddsportal[df_oddsportal.Name == op_tour]['url'].iloc[0]
    df_odds = scrape_oddsportal_fixed(url_op, headless=False)
    df_odds['tournament_op'] = op_tour
    df_odds_super = pd.concat([df_odds_super, df_odds], ignore_index=True)

display(df_odds_super)

In [ ]:
# Save odds super

df_odds_super['Date'] = pd.to_datetime(df_odds_super['Date'])
df_odds_super.sort_values('Date', inplace=True)
df_odds_super = df_odds_super.reset_index(drop=True)
display(df_odds_super)

df_odds_super.to_csv('data/odds.csv', index=False)

## Fuzzy match

In [ ]:
# Fuzzy matching JSON

import pandas as pd
from fuzzywuzzy import process
import json
from pathlib import Path
import re

# --- Helper függvény: normalizálás ---
def normalize_team_name(name):
    name = name.lower()
    # eltávolítjuk a 'team', 'gaming', 'esports' szavakat
    name = re.sub(r'\b(team|gaming|esports)\b', '', name)
    # extra whitespace eltávolítás
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# --- Egyedi csapatnevek ---
odds_teams = pd.unique(df_odds_super[['home_team','away_team']].values.ravel())
match_teams = pd.unique(matches[['team1','team2']].values.ravel())

# --- Normalizált változatok készítése fuzzyhoz ---
normalized_match_teams = {normalize_team_name(t): t for t in match_teams}

team_mapping = {}
for team in odds_teams:
    norm_team = normalize_team_name(team)
    
    # fuzzy match a normalizált neveken
    best_match_norm, score = process.extractOne(norm_team, normalized_match_teams.keys())
    # visszakapjuk az eredeti matches csapatnevet
    matched_team = normalized_match_teams[best_match_norm]
    
    team_mapping[team] = {
        "matched_team": matched_team,
        "score": score
    }

# --- Mentés JSON-ba ---
#Path("data").mkdir(exist_ok=True)
#with open("data/team_name_mapping.json", "w", encoding="utf-8") as f:
#    json.dump(team_mapping, f, indent=2, ensure_ascii=False)

#print("✅ Normalizált fuzzy team mapping kész, elmentve: data/team_name_mapping.json")


In [ ]:
# Map team names

import json

with open("data/team_name_mapping.json", "r", encoding="utf-8") as f:
    team_mapping = json.load(f)

df_odds_super = pd.read_csv('data/odds.csv')

for i, row in df_odds_super.iterrows():
    df_odds_super.loc[i, 'team1'] = team_mapping[row['home_team']]['matched_team']
    df_odds_super.loc[i, 'team2'] = team_mapping[row['away_team']]['matched_team']


# Matches mapping
## convert name and type
df_odds_super.rename(columns={'Date': 'date'}, inplace=True)
df_odds_super['date'] = pd.to_datetime(df_odds_super['date'])

## merge manually
for i, row in df_odds_super.iterrows():
    home_op = row['team1']
    away_op = row['team2']

    rel_matches = matches[(matches.date < row['date'] + pd.Timedelta(days=1)) & 
                          (matches.date > row['date'] - pd.Timedelta(days=1))]
    if rel_matches.shape[0] == 0:
        continue
    else:
        for j, match in rel_matches.iterrows():
            if home_op == match['team1'] and away_op == match['team2']:
                matches.loc[j, 'team1_odds'] = row['home_odds']
                matches.loc[j, 'team2_odds'] = row['away_odds']
            elif home_op == match['team2'] and away_op == match['team1']:
                matches.loc[j, 'team1_odds'] = row['away_odds']
                matches.loc[j, 'team2_odds'] = row['home_odds']

matches_o = matches[matches.team1_odds.notna()].reset_index(drop=True).copy()
display(matches_o.sample(7))
print(f"Matches with matched odds: {len(matches_o)}/{len(matches)}")

# ML

In [ ]:
# ML features

import numpy as np

# --- 1️⃣ Tournament info integrálása matches_o-ba ---
ml_matches = matches.copy()

# Merge tournament metadata
ml_matches = pd.merge(ml_matches,
    tournaments[['tournament_name', 'region', 'split', 'stage']],
    how='left',
    on='tournament_name'
)

# --- 2️⃣ Elo alapú feature-ök ---
ml_matches['elo_diff'] = ml_matches['team1_elo_before'] - ml_matches['team2_elo_before']

# --- 3️⃣ Forma / momentum feature-ök ---
ml_matches = ml_matches.sort_values('date')
ml_matches['team1_win'] = (ml_matches['team1_score'] > ml_matches['team2_score']).astype(int)
ml_matches['team2_win'] = (ml_matches['team2_score'] > ml_matches['team1_score']).astype(int)

# Rolling window 5 meccs soronként
def rolling_winrate_rowwise(df, n=5):
    team1_winrates = []
    team2_winrates = []

    df_sorted = df.sort_values('date')

    for idx, row in df_sorted.iterrows():
        team1 = row['team1']
        team2 = row['team2']
        date = row['date']

        # --- Team1 winrate ---
        past_team1 = df_sorted[
            ((df_sorted['team1'] == team1) | (df_sorted['team2'] == team1)) &
            (df_sorted['date'] < date)
        ].sort_values('date', ascending=False).head(n)
        if len(past_team1) == 0:
            team1_winrates.append(np.nan)
        else:
            wins = sum(
                (m['team1_win'] if m['team1'] == team1 else m['team2_win'])
                for _, m in past_team1.iterrows()
            )
            team1_winrates.append(wins / len(past_team1))

        # --- Team2 winrate ---
        past_team2 = df_sorted[
            ((df_sorted['team1'] == team2) | (df_sorted['team2'] == team2)) &
            (df_sorted['date'] < date)
        ].sort_values('date', ascending=False).head(n)
        if len(past_team2) == 0:
            team2_winrates.append(np.nan)
        else:
            wins = sum(
                (m['team1_win'] if m['team1'] == team2 else m['team2_win'])
                for _, m in past_team2.iterrows()
            )
            team2_winrates.append(wins / len(past_team2))

    return team1_winrates, team2_winrates

ml_matches['team1_winrate_3'], ml_matches['team2_winrate_3'] = rolling_winrate_rowwise(ml_matches, n=3)
ml_matches['team1_winrate_5'], ml_matches['team2_winrate_5'] = rolling_winrate_rowwise(ml_matches, n=5)
ml_matches['team1_winrate_10'], ml_matches['team2_winrate_10'] = rolling_winrate_rowwise(ml_matches, n=10)

# --- 4️⃣ BO / patch / stage feature-ök ---
ml_matches['patch'] = ml_matches['patch'].astype(str)
ml_matches['is_playoffs'] = ml_matches['tournament_name'].str.contains('Playoffs|Finals|Championship', case=False).astype(int)
# --- BO type visszafejtése ---
max_score = ml_matches[['team1_score', 'team2_score']].max(axis=1)

def infer_bo(m):
    if m == 1: 
        return 1  # BO1
    elif m == 2: 
        return 3  # BO3
    elif m >= 3: 
        return 5  # BO5
    return 1  # fallback, ha valami edge-case lenne

ml_matches['bo_type'] = max_score.apply(infer_bo)

# --- 5️⃣ Games statisztika aggregálás (match szintre) ---
def assign_team_sides(row, team1, team2):
    """
    Kiszámolja a statisztikákat a megadott csapatra,
    függetlenül attól, hogy blue vagy red oldalon játszott.
    """
    result = {}
    
    # HA A CSAPAT BLUE OLDALON VOLT
    if row['blue_team'] == team1:
        result['team1_gold_diff_15min'] = row['gold_diff_15min']
        result['team1_first_blood'] = int(row['blue_first_blood'])
        result['team1_first_tower'] = int(row['blue_first_tower'])
        result['team1_barons'] = row['blue_barons']
        result['team1_dragons'] = row['blue_dragons']
        result['team1_wards_destroyed'] = row['vision_team1_wards_destroyed']
        result['team1_wards_placed'] = row['vision_team1_wards_placed']
    # HA A CSAPAT RED OLDALON VOLT
    elif row['red_team'] == team1:
        result['team1_gold_diff_15min'] = -row['gold_diff_15min']
        result['team1_first_blood'] = int(row['red_first_blood'])
        result['team1_first_tower'] = int(row['red_first_tower'])
        result['team1_barons'] = row['red_barons']
        result['team1_dragons'] = row['red_dragons']
        result['team1_wards_destroyed'] = row['vision_team2_wards_destroyed']
        result['team1_wards_placed'] = row['vision_team2_wards_placed']
    else:
        # Ha a csapat nem szerepel ebben a game-ben
        result = {f'team1_{k}': 0 for k in ['gold_diff_15min', 'first_blood', 'first_tower', 'barons', 'dragons', 'wards_destroyed', 'wards_placed']}
    
    # UGYANEZ TEAM2-RE (ha kell)
    if team2 != 'dummy':  # dummy jelzi, hogy csak team1-et számolunk
        if row['blue_team'] == team2:
            result['team2_gold_diff_15min'] = row['gold_diff_15min']
            result['team2_first_blood'] = int(row['blue_first_blood'])
            result['team2_first_tower'] = int(row['blue_first_tower'])
            result['team2_barons'] = row['blue_barons']
            result['team2_dragons'] = row['blue_dragons']
            result['team2_wards_destroyed'] = row['vision_team1_wards_destroyed']
            result['team2_wards_placed'] = row['vision_team1_wards_placed']
        elif row['red_team'] == team2:
            result['team2_gold_diff_15min'] = -row['gold_diff_15min']
            result['team2_first_blood'] = int(row['red_first_blood'])
            result['team2_first_tower'] = int(row['red_first_tower'])
            result['team2_barons'] = row['red_barons']
            result['team2_dragons'] = row['red_dragons']
            result['team2_wards_destroyed'] = row['vision_team2_wards_destroyed']
            result['team2_wards_placed'] = row['vision_team2_wards_placed']
        else:
            result.update({f'team2_{k}': 0 for k in ['gold_diff_15min', 'first_blood', 'first_tower', 'barons', 'dragons', 'wards_destroyed', 'wards_placed']})
    
    return pd.Series(result)

# Merge minden match_id-re - CSAK MÚLTBELI MECCSEK
all_features = []
n_past_matches = 3  # hány korábbi meccset átlagolsz

for idx, match in ml_matches.iterrows():
    team1 = match['team1']
    team2 = match['team2']
    current_date = match['date']
    
    # ✅ TEAM1 múltbeli game-jei (n darab meccs)
    past_matches_team1 = ml_matches[
        ((ml_matches['team1'] == team1) | (ml_matches['team2'] == team1)) &
        (ml_matches['date'] < current_date)  # ✅ CSAK MÚLT
    ].sort_values('date', ascending=False).head(n_past_matches)
    past_games_team1 = games[games['match_id'].isin(past_matches_team1['match_id'])]
    
    # ✅ TEAM1 ellenfelek ELO (átlag + szórás)
    if len(past_matches_team1) > 0:
        # ellenfél = ha team1 volt team1 -> team2_elo_before, ha team1 volt team2 -> team1_elo_before
        opponent_elos_team1 = past_matches_team1.apply(
            lambda m: m['team2_elo_before'] if m['team1'] == team1 else m['team1_elo_before'],
            axis=1
        )
        team1_opp_elo_mean = opponent_elos_team1.mean()
        team1_opp_elo_std = opponent_elos_team1.std()
        
        opponent_elodiffs_team1 = past_matches_team1.apply(
            lambda m: m['team2_elo_before'] - m['team1_elo_before'] if m['team1'] == team1 else m['team1_elo_before'] - m['team2_elo_before'],
            axis=1
        )
        team1_opp_elodiff_mean = opponent_elodiffs_team1.mean()
        team1_opp_elodiff_std = opponent_elodiffs_team1.std()
    else:
        team1_opp_elo_mean = 0
        team1_opp_elo_std = 0
        team1_opp_elodiff_mean = 0
        team1_opp_elodiff_std = 0

    # ✅ TEAM2 múltbeli game-jei
    past_matches_team2 = ml_matches[
        ((ml_matches['team1'] == team2) | (ml_matches['team2'] == team2)) &
        (ml_matches['date'] < current_date)
    ].sort_values('date', ascending=False).head(n_past_matches)
    
    past_games_team2 = games[games['match_id'].isin(past_matches_team2['match_id'])]
    
    # ✅ TEAM2 ellenfelek ELO (átlag + szórás)
    if len(past_matches_team2) > 0:
        opponent_elos_team2 = past_matches_team2.apply(
            lambda m: m['team2_elo_before'] if m['team1'] == team2 else m['team1_elo_before'],
            axis=1
        )
        team2_opp_elo_mean = opponent_elos_team2.mean()
        team2_opp_elo_std = opponent_elos_team2.std()

        opponent_elodiffs_team2 = past_matches_team2.apply(
            lambda m: m['team2_elo_before'] - m['team1_elo_before'] if m['team1'] == team2 else m['team1_elo_before'] - m['team2_elo_before'],
            axis=1
        )
        team2_opp_elodiff_mean = opponent_elodiffs_team2.mean()
        team2_opp_elodiff_std = opponent_elodiffs_team2.std()
    else:
        team2_opp_elo_mean = 0
        team2_opp_elo_std = 0
        team2_opp_elodiff_mean = 0
        team2_opp_elodiff_std = 0

    # Aggregálás team1-re
    if len(past_games_team1) > 0:
        temp1 = past_games_team1.apply(assign_team_sides, axis=1, team1=team1, team2='dummy')
        team1_stats = temp1.mean()
    else:
        team1_stats = pd.Series({col: 0 for col in [
            'team1_gold_diff_15min', 'team1_first_blood', 'team1_first_tower',
            'team1_barons', 'team1_dragons', 'team1_wards_destroyed', 'team1_wards_placed'
        ]})
    
    # Aggregálás team2-re
    if len(past_games_team2) > 0:
        temp2 = past_games_team2.apply(assign_team_sides, axis=1, team1='dummy', team2=team2)
        team2_stats = temp2.mean()
    else:
        team2_stats = pd.Series({col: 0 for col in [
            'team2_gold_diff_15min', 'team2_first_blood', 'team2_first_tower',
            'team2_barons', 'team2_dragons', 'team2_wards_destroyed', 'team2_wards_placed'
        ]})
    
    # Kombinálás - DICT-KÉ ALAKÍTÁS
    combined_dict = {**team1_stats.to_dict(), **team2_stats.to_dict()}
    combined_dict['team1_opponent_elo_mean'] = team1_opp_elo_mean
    combined_dict['team1_opponent_elo_std'] = team1_opp_elo_std
    combined_dict['team2_opponent_elo_mean'] = team2_opp_elo_mean
    combined_dict['team2_opponent_elo_std'] = team2_opp_elo_std
    combined_dict['team1_opponent_elodiff_mean'] = team1_opp_elodiff_mean
    combined_dict['team1_opponent_elodiff_std'] = team1_opp_elodiff_std
    combined_dict['team2_opponent_elodiff_mean'] = team2_opp_elodiff_mean
    combined_dict['team2_opponent_elodiff_std'] = team2_opp_elodiff_std
    combined_dict['match_id'] = match['match_id']
    all_features.append(combined_dict)

games_agg_corrected = pd.DataFrame(all_features)
match_agg_corrected = games_agg_corrected.groupby('match_id', as_index=False).mean()

ml_matches = ml_matches.merge(match_agg_corrected, on='match_id', how='left')

# --- 6️⃣ Vision és objective control arányok ---
ml_matches['team1_vision_control'] = ml_matches['team1_wards_destroyed'] / (ml_matches['team1_wards_placed'] + 1)
ml_matches['team2_vision_control'] = ml_matches['team2_wards_destroyed'] / (ml_matches['team2_wards_placed'] + 1)

ml_matches['team1_dragon_control'] = ml_matches['team1_dragons'] / (ml_matches['team1_dragons'] + ml_matches['team2_dragons'] + 1)
ml_matches['team2_dragon_control'] = ml_matches['team2_dragons'] / (ml_matches['team1_dragons'] + ml_matches['team2_dragons'] + 1)

ml_matches['team1_baron_control'] = ml_matches['team1_barons'] / (ml_matches['team1_barons'] + ml_matches['team2_barons'] + 1)
ml_matches['team2_baron_control'] = ml_matches['team2_barons'] / (ml_matches['team1_barons'] + ml_matches['team2_barons'] + 1)

# --- 7️⃣ Kimenetel target --- 
ml_matches['target'] = (ml_matches['team1_score'] > ml_matches['team2_score']).astype(int)

# Enhanced features
import pandas as pd
import numpy as np

def add_enhanced_features(ml_matches, matches, games):
    """
    Új feature-ök hozzáadása a meglévő ml_matches DataFrame-hez
    """
    
    # ===== 1. HEAD-TO-HEAD STATISTICS =====
    def calculate_h2h_features(row):
        team1, team2 = row['team1'], row['team2']
        current_date = row['date']
        
        # Korábbi H2H meccsek
        h2h_matches = matches[
            (((matches['team1'] == team1) & (matches['team2'] == team2)) |
             ((matches['team1'] == team2) & (matches['team2'] == team1))) &
            (matches['date'] < current_date)
        ].sort_values('date', ascending=False).head(5)
        
        if len(h2h_matches) == 0:
            return pd.Series({
                'h2h_winrate': 0.5,
                'h2h_games_played': 0,
                'h2h_avg_score_diff': 0
            })
        
        # Team1 győzelmek számítása
        wins = sum(
            (m['team1_score'] > m['team2_score'] and m['team1'] == team1) or
            (m['team2_score'] > m['team1_score'] and m['team2'] == team1)
            for _, m in h2h_matches.iterrows()
        )
        
        # Átlagos score különbség
        score_diffs = [
            (m['team1_score'] - m['team2_score'] if m['team1'] == team1 
             else m['team2_score'] - m['team1_score'])
            for _, m in h2h_matches.iterrows()
        ]
        
        return pd.Series({
            'h2h_winrate': wins / len(h2h_matches),
            'h2h_games_played': len(h2h_matches),
            'h2h_avg_score_diff': np.mean(score_diffs)
        })
    
    h2h_features = ml_matches.apply(calculate_h2h_features, axis=1)
    ml_matches = pd.concat([ml_matches, h2h_features], axis=1)
    
    
    # ===== 2. TIME-BASED FORM =====
    def calculate_time_based_form(df, team, date, days=30):
        recent_matches = df[
            (((df['team1'] == team) | (df['team2'] == team)) &
             (df['date'] < date) &
             (df['date'] >= date - pd.Timedelta(days=days)))
        ]
        
        if len(recent_matches) == 0:
            return pd.Series({'winrate': np.nan, 'games_count': 0})
        
        wins = sum(
            (m['team1_win'] if m['team1'] == team else m['team2_win'])
            for _, m in recent_matches.iterrows()
        )
        
        return pd.Series({
            'winrate': wins / len(recent_matches),
            'games_count': len(recent_matches)
        })
    
    # 30 napos forma
    form_30d_team1 = ml_matches.apply(
        lambda r: calculate_time_based_form(ml_matches, r['team1'], r['date'], 30),
        axis=1
    )
    form_30d_team2 = ml_matches.apply(
        lambda r: calculate_time_based_form(ml_matches, r['team2'], r['date'], 30),
        axis=1
    )
    
    ml_matches['team1_winrate_30d'] = form_30d_team1['winrate']
    ml_matches['team1_games_30d'] = form_30d_team1['games_count']
    ml_matches['team2_winrate_30d'] = form_30d_team2['winrate']
    ml_matches['team2_games_30d'] = form_30d_team2['games_count']
    
    
    # ===== 3. ELO MOMENTUM =====
    def calculate_elo_momentum(df, team, date, n=5):
        recent = df[
            (((df['team1'] == team) | (df['team2'] == team)) &
             (df['date'] < date))
        ].sort_values('date', ascending=False).head(n)
        
        if len(recent) < 2:
            return 0
        
        elo_changes = [
            (m['team1_elo_after'] - m['team1_elo_before'] if m['team1'] == team 
             else m['team2_elo_after'] - m['team2_elo_before'])
            for _, m in recent.iterrows()
        ]
        
        return np.mean(elo_changes)
    
    ml_matches['team1_elo_momentum'] = ml_matches.apply(
        lambda r: calculate_elo_momentum(ml_matches, r['team1'], r['date']),
        axis=1
    )
    ml_matches['team2_elo_momentum'] = ml_matches.apply(
        lambda r: calculate_elo_momentum(ml_matches, r['team2'], r['date']),
        axis=1
    )
    
    
    # ===== 4. BLUE/RED SIDE WINRATES =====
    def calculate_side_winrate(games_df, matches_df, team, date, side='blue'):
        # Matches IDs where team played
        team_matches = matches_df[
            (((matches_df['team1'] == team) | (matches_df['team2'] == team)) &
             (matches_df['date'] < date))
        ]['match_id']
        
        # Games on specific side
        if side == 'blue':
            side_games = games_df[
                (games_df['match_id'].isin(team_matches)) &
                (games_df['blue_team'] == team)
            ]
            wins = (side_games['blue_result'] == 'WIN').sum()
        else:
            side_games = games_df[
                (games_df['match_id'].isin(team_matches)) &
                (games_df['red_team'] == team)
            ]
            wins = (side_games['red_result'] == 'WIN').sum()
        
        if len(side_games) == 0:
            return np.nan
        
        return wins / len(side_games)
    
    ml_matches['team1_blue_winrate'] = ml_matches.apply(
        lambda r: calculate_side_winrate(games, matches, r['team1'], r['date'], 'blue'),
        axis=1
    )
    ml_matches['team1_red_winrate'] = ml_matches.apply(
        lambda r: calculate_side_winrate(games, matches, r['team1'], r['date'], 'red'),
        axis=1
    )
    ml_matches['team2_blue_winrate'] = ml_matches.apply(
        lambda r: calculate_side_winrate(games, matches, r['team2'], r['date'], 'blue'),
        axis=1
    )
    ml_matches['team2_red_winrate'] = ml_matches.apply(
        lambda r: calculate_side_winrate(games, matches, r['team2'], r['date'], 'red'),
        axis=1
    )
    
    
    # ===== 5. DAYS SINCE LAST MATCH =====
    def days_since_last_match(df, team, date):
        last_match = df[
            (((df['team1'] == team) | (df['team2'] == team)) &
             (df['date'] < date))
        ].sort_values('date', ascending=False).head(1)
        
        if len(last_match) == 0:
            return np.nan
        
        return (date - last_match.iloc[0]['date']).days
    
    ml_matches['team1_days_rest'] = ml_matches.apply(
        lambda r: days_since_last_match(ml_matches, r['team1'], r['date']),
        axis=1
    )
    ml_matches['team2_days_rest'] = ml_matches.apply(
        lambda r: days_since_last_match(ml_matches, r['team2'], r['date']),
        axis=1
    )
    
    
    # ===== 6. BETTING MARKET FEATURES =====
    # Implied probability from odds
    ml_matches['team1_implied_prob'] = 1 / ml_matches['team1_odds']
    ml_matches['team2_implied_prob'] = 1 / ml_matches['team2_odds']
    
    # Market margin (bookie confidence)
    ml_matches['market_margin'] = (
        ml_matches['team1_implied_prob'] + ml_matches['team2_implied_prob'] - 1
    )
    
    # Odds ratio (underdog indicator)
    ml_matches['odds_ratio'] = ml_matches['team1_odds'] / ml_matches['team2_odds']
    
    
    # ===== 7. TOURNAMENT PROGRESSION =====
    def games_in_tournament(df, team, tournament, date):
        return len(df[
            (((df['team1'] == team) | (df['team2'] == team)) &
             (df['tournament_name'] == tournament) &
             (df['date'] < date))
        ])
    
    ml_matches['team1_tournament_games'] = ml_matches.apply(
        lambda r: games_in_tournament(ml_matches, r['team1'], r['tournament_name'], r['date']),
        axis=1
    )
    ml_matches['team2_tournament_games'] = ml_matches.apply(
        lambda r: games_in_tournament(ml_matches, r['team2'], r['tournament_name'], r['date']),
        axis=1
    )
    
    
    # Fill NaNs
    ml_matches = ml_matches.fillna(0)
    
    return ml_matches


# Használat:
ml_matches = add_enhanced_features(ml_matches, matches, games)

# --- 8️⃣ ML input kész ---
ml_features = [
    # === CORE ELO FEATURES (megtartva) ===
    'team1_elo_before', 'team2_elo_before', 'elo_diff',
    
    # === HEAD-TO-HEAD (ÚJ - nagyon erős!) ===
    'h2h_winrate',
    'h2h_games_played',
    'h2h_avg_score_diff',
    
    # === RECENT FORM - Mixed time windows ===
    'team1_winrate_3', 'team2_winrate_3',      # Legutóbbi forma
    'team1_winrate_10', 'team2_winrate_10',    # Hosszabb távú
    'team1_winrate_30d', 'team2_winrate_30d',  # ÚJ - időalapú (küszöböli ki a long pause hatást)
    
    # === ELO CONTEXT & MOMENTUM (ÚJ) ===
    'team1_elo_momentum', 'team2_elo_momentum',  # Emelkedő/csökkenő forma
    'team1_opponent_elo_mean', 'team2_opponent_elo_mean',  # Schedule strength
    
    # === BETTING MARKET SIGNALS (ÚJ - fontos!) ===
    #'team1_implied_prob', 'team2_implied_prob',  # Bookmaker értékelés
    'market_margin',                              # Bookie bizonyosság
    #'odds_ratio',                                 # Underdog indikátor
    
    # === MATCH CONTEXT ===
    'bo_type',                                    # BO1/BO3/BO5
    'is_playoffs',                                # Magasabb tét
    'is_international',                           # Liga szint
    'team1_tournament_games', 'team2_tournament_games',  # ÚJ - Tournament tapasztalat
    
    # === SIDE PREFERENCE (ÚJ - LoL specifikus!) ===
    'team1_blue_winrate', 'team1_red_winrate',
    'team2_blue_winrate', 'team2_red_winrate',
    
    # === REST & FATIGUE (ÚJ) ===
    'team1_days_rest', 'team2_days_rest',
    'team1_games_30d', 'team2_games_30d',        # Terhelés indikátor
    
    # === IN-GAME PERFORMANCE (legjobb historikus average-ek) ===
    'team1_vision_control', 'team2_vision_control',
    'team1_dragon_control', 'team2_dragon_control',
    'team1_baron_control', 'team2_baron_control'
]

ml_df = ml_matches[['target', 'match_id', 'date', 'tournament_name', 'team1', 'team2'] + 
                   ml_features].copy()
ml_df.fillna(0, inplace=True)

print("✅ ML input kész, shape:", ml_df.shape)
display(ml_df.tail())

In [ ]:
# ML betting sim with TimeSeriesSplit

import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score

# --- Modellek importálása ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

# --- Settings ---
N_SPLITS = 5
VALUE_MIN, VALUE_MAX = 0.03, 0.50
MIN_CONFIDENCE, MAX_CONFIDENCE = 0.55, 0.95

# --- 1) Csak azokat a meccseket tartjuk meg, amelyekhez vannak oddsok ---
ml_df_odds = ml_df[ml_df['match_id'].isin(matches_o['match_id'])].copy()

# --- 2) Merge odds ---
ml_df_odds = ml_df_odds.merge(
    matches_o[['match_id', 'team1_odds', 'team2_odds']],
    on='match_id',
    how='left'
)

# --- Rendezés idő szerint ---
ml_df_odds = ml_df_odds.sort_values('date').reset_index(drop=True)

# --- Features / target ---
features = ml_features.copy()

X = ml_df_odds[features].copy()
y = ml_df_odds['target']

# --- Tesztelendő modellek ---
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, eval_metric='logloss', random_state=42)
}


# --- TimeSeriesSplit ---
tscv = TimeSeriesSplit(n_splits=N_SPLITS)

model_results = {}
for model_name, model in models.items():
    model_results[model_name] = {}
    fold_id = 1
    
    for train_idx, test_idx in tscv.split(X):
        model_results[model_name][f"fold{fold_id}"] = {}
        # Train / test split foldonként
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        test_df = ml_df_odds.iloc[test_idx].copy()

        # Modell tanítása
        model.fit(X_train, y_train)

        # Accuracy foldra
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        # Value betting szimuláció foldra
        bankroll = 30.0
        unit = 1.0
        bets = 0

        for idx, row in test_df.iterrows():
            X_row = row[features].values.reshape(1, -1)
            p_team1 = model.predict_proba(X_row)[0][1]
            p_team2 = 1 - p_team1

            # Margin számítás
            value1 = row['team1_odds'] * p_team1 - 1
            value2 = row['team2_odds'] * p_team2 - 1

            # Csak VALUE_MIN < margin < VALUE_MAX
            if (VALUE_MIN < value1 < VALUE_MAX) and (MIN_CONFIDENCE < p_team1 < MAX_CONFIDENCE):
                bets += 1
                win = row['target'] == 1
                bankroll += unit * (row['team1_odds'] - 1) if win else -unit

            if (VALUE_MIN < value2 < VALUE_MAX) and (MIN_CONFIDENCE < p_team2 < MAX_CONFIDENCE):
                bets += 1
                win = row['target'] == 0
                bankroll += unit * (row['team2_odds'] - 1) if win else -unit

        roi = (bankroll - 30) / 30 * 100

        model_results[model_name][f"fold{fold_id}"]['acc'] =  acc
        model_results[model_name][f"fold{fold_id}"]['roi'] = roi
        model_results[model_name][f"fold{fold_id}"]['bets'] = bets
        
        fold_id += 1

In [ ]:
# ML model results summary

for model_name, fold_results in model_results.items():
    print()
    print("="*60)
    print(f"Model: {model_name}")
    for fold, result in fold_results.items():
        print(f"{fold}: Accuracy: {result['acc']:.3f}, ROI: {result['roi']:.1f}%, Value bets: {result['bets']}")
    
    print(f"\nAverage Accuracy: {np.mean([result['acc'] for result in fold_results.values()]):.3f}")
    print(f"Accuracy std: {np.std([result['acc'] for result in fold_results.values()]):.3f}")
    print(f"Average ROI: {np.mean([result['roi'] for result in fold_results.values()]):.1f}%")
    print(f"ROI std: {np.std([result['roi'] for result in fold_results.values()]):.1f}pp")

In [ ]:
# Save NeuralNet

best_model = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
best_model.fit(X, y)

import joblib
joblib.dump(best_model, "models/neural_net_model_20251029.pkl")


In [ ]:
# vs Baseline: favorit bet

import pandas as pd

# Feltételezve, hogy az oddsok már benne vannak ml_df-ben: 'team1_odds', 'team2_odds'
# És a target: 1 ha team1 nyer, 0 ha team2

# 1️⃣ Kiszámoljuk a favorit csapatot minden sorban
def get_favorite(row):
    if row['team1_odds'] < row['team2_odds']:
        return 'team1'
    else:
        return 'team2'

ml_df_odds['favorite'] = ml_df_odds.apply(get_favorite, axis=1)

# 2️⃣ Kimenetel ellenőrzése a favoritra
def favorite_win(row):
    if row['favorite'] == 'team1':
        return row['target'] == 1
    else:
        return row['target'] == 0

ml_df_odds['favorite_win'] = ml_df_odds.apply(favorite_win, axis=1)

# 3️⃣ Szimuláció
bankroll_bl = 30  # kezdő bankroll
bet_unit = 1
wins = 0
for idx, row in ml_df_odds.iterrows():
    odds = row['team1_odds'] if row['favorite'] == 'team1' else row['team2_odds']
    if row['favorite_win']:
        bankroll_bl += bet_unit * (odds - 1)  # profit
        wins += 1
    else:
        bankroll_bl -= bet_unit  # veszteség

# 4️⃣ Eredmények
accuracy_bl = ml_df_odds['favorite_win'].mean()
roi_bl = (bankroll_bl - 30) / 30 * 100  # ROI százalékban

print(f"Favorite betting count: {len(ml_df_odds)}")
print(f"Wins: {wins}")
print(f"Final bankroll: {bankroll_bl:.2f} (model: {bankroll:.2f})")
print(f"Accuracy: {accuracy_bl:.3f} (model: {acc:.2f})")
print(f"ROI: {roi_bl:.0f}% (model: {roi:.0f}%)")


In [ ]:
# vs ELO bet

import pandas as pd
import numpy as np

# Feltételezve, hogy ml_df_odds tartalmaz: 'team1_elo_before', 'team2_elo_before', 'team1_odds', 'team2_odds', 'target'

# 1️⃣ ELO alapú win valószínűség
def elo_win_prob(row):
    return 1 / (1 + 10 ** ((row['team2_elo_before'] - row['team1_elo_before']) / 400))

ml_df_odds['team1_prob'] = ml_df_odds.apply(elo_win_prob, axis=1)
ml_df_odds['team2_prob'] = 1 - ml_df_odds['team1_prob']

# 2️⃣ Value bet logika
def value_bet(row):
    # team1 value bet?
    if row['team1_odds'] * row['team1_prob'] > 1:
        return 'team1'
    elif row['team2_odds'] * row['team2_prob'] > 1:
        return 'team2'
    else:
        return None

ml_df_odds['value_bet'] = ml_df_odds.apply(value_bet, axis=1)

# 3️⃣ Szimuláció
bankroll_vb = 30
bet_unit = 1
wins_vb = 0
value_bets_count = 0

for idx, row in ml_df_odds.iterrows():
    if row['value_bet'] is not None:
        value_bets_count += 1
        odds = row['team1_odds'] if row['value_bet'] == 'team1' else row['team2_odds']
        won = (row['target'] == 1 and row['value_bet'] == 'team1') or \
              (row['target'] == 0 and row['value_bet'] == 'team2')
        if won:
            bankroll_vb += bet_unit * (odds - 1)
            wins_vb += 1
        else:
            bankroll_vb -= bet_unit

# 4️⃣ Eredmények
accuracy_vb = wins_vb / value_bets_count if value_bets_count > 0 else np.nan
roi_vb = (bankroll_vb - 30) / 30 * 100

print(f"✅ Value bets count: {value_bets_count}")
print(f"✅ Wins: {wins_vb}")
print(f"✅ Final bankroll: {bankroll_vb:.2f}")
print(f"✅ Accuracy: {accuracy_vb:.3f}")
print(f"✅ ROI: {roi_vb:.0f}%")


In [ ]:
# Feature importance

import pandas as pd
import matplotlib.pyplot as plt

# Feltételezve, hogy a modell neve rf és a feature lista X.columns
importances = model.feature_importances_
feature_names = X.columns

# DataFrame-be pakoljuk
feat_imp = pd.DataFrame({'feature': feature_names, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False)

# Kiíratás
print(feat_imp)

# Vizu
plt.figure(figsize=(10,6))
plt.barh(feat_imp['feature'], feat_imp['importance'])
plt.gca().invert_yaxis()  # a legfontosabb fent legyen
plt.xlabel('Importance')
plt.title('Random Forest Feature Importances')
plt.show()

